HYPERPARAMETER TUNING CON KERAS TUNER

Keras Tuned serve a trovare automaticamente buoni iperparametri per un modello Keras: numero di neuroni, layer, learning rate, batch size, ecc. Non allena il modello migliore in assoluto. Cerca tra combinazioni che tu gli dai. Se gli dai uno spazio di ricerca stupido, trova il migliore tra le scelte stupide.

In [ ]:
#pip install keras-tuner


   -------------------- ------------------- 1/2 [keras-tuner]
   -------------------- ------------------- 1/2 [keras-tuner]
   -------------------- ------------------- 1/2 [keras-tuner]
   -------------------- ------------------- 1/2 [keras-tuner]
   ---------------------------------------- 2/2 [keras-tuner]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt

# =========================
# 1. CREAZIONE DATI FITTIZI
# =========================

np.random.seed(42)

df = pd.DataFrame({
    "lunghezza": np.random.randint(10, 200, 500),
    "larghezza": np.random.randint(10, 100, 500),
    "altezza": np.random.randint(1, 50, 500),
    "materia_prima_kg": np.random.uniform(0.1, 20, 500),
})

# Formula fittizia del peso
df["peso"] = (
    df["lunghezza"] * 0.02 +
    df["larghezza"] * 0.03 +
    df["altezza"] * 0.05 +
    df["materia_prima_kg"] * 0.8 +
    np.random.normal(0, 1, 500)
)

# =========================
# 2. FEATURE E TARGET
# =========================

X = df[[
    "lunghezza",
    "larghezza",
    "altezza",
    "materia_prima_kg"
]]

y = df["peso"]

# =========================
# 3. TRAIN TEST SPLIT
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# =========================
# 4. NORMALIZZAZIONE
# =========================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# =========================
# 5. MODELLO KERAS TUNER
# =========================

def build_model(hp):

    model = keras.Sequential()

    model.add(layers.Input(shape=(X_train.shape[1],)))

    # Numero neuroni variabile
    model.add(
        layers.Dense(
            units=hp.Int(
                "neuroni",
                min_value=16,
                max_value=128,
                step=16
            ),
            activation="relu"
        )
    )

    # Dropout variabile
    model.add(
        layers.Dropout(
            hp.Float(
                "dropout",
                min_value=0.0,
                max_value=0.5,
                step=0.1
            )
        )
    )

    model.add(layers.Dense(1))

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=hp.Choice(
                "learning_rate",
                values=[0.01, 0.001, 0.0001]
            )
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model

# =========================
# 6. TUNER
# =========================

tuner = kt.RandomSearch(
    build_model,
    objective="val_mae",
    max_trials=10,
    overwrite=True,
    directory="tuning",
    project_name="peso_articoli"
)

# =========================
# 7. TRAINING
# =========================

tuner.search(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32
)

# =========================
# 8. MIGLIOR MODELLO
# =========================

best_model = tuner.get_best_models(num_models=1)[0]

best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("\nMigliori iperparametri:")
print(best_hp.values)

# =========================
# 9. VALUTAZIONE
# =========================

loss, mae = best_model.evaluate(X_test, y_test)

print(f"\nMAE finale: {mae:.2f}")

Trial 10 Complete [00h 00m 06s]
val_mae: 1.100403070449829

Best val_mae So Far: 0.9979100227355957
Total elapsed time: 00h 00m 59s


c:\Users\uberti\.conda\envs\ai_epicode\Lib\site-packages\keras\src\saving\saving_lib.py:801: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))



Migliori iperparametri:
{'neuroni': 96, 'dropout': 0.30000000000000004, 'learning_rate': 0.01}
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 1.8212 - mae: 1.0990 

MAE finale: 1.10


poi lanci la ricerca

Keras Tuned prova più versioni del modello, guarda quale ha il migliore val_mae, restituisce i parametri vincenti.

Ma attenzione, se i dati di partenza sono sporchi, Keras Tuned non migliora la situazione.

Keras Tunes: accorda automaticamente i modelli.

- Definizione dello spazio di ricerca
- Strategia di ricerca e confronto
- Verdetto finale, estrazione e validazione del modello vincente

Perchè dobbiao abbandonare i valori statici
Finora abbiamo costruito neuroni con un numero fisso di neuroni e un learning rate scelto ad intuito (0.01). Tuttavia, trovare la combinazione perfeeta manualmente è spesso frustrante e inefficiente.
Ogni dataset è un pò un foresta diversa e richiede una mappa specifica. 
Creiamo modelli elastici, pronti ad adattarsi, con l'oggetto hyperparameters

L'oggetto HyperParameters
- Il metodo hp.init permette di definire un intervallo di numero interi per il numero di neuroni, specificando un valore minimo uno massimo e uno step incrementale. Decidiamo la largehzza dei layer, non più un valore fisso ma un intervallo dove il tuned può scorrere.
- Il parametro hp.Float è ideale per parametri continui come il learning rate, permettendo spesso l'uso di una scala logaritmica per esplorare i diversi ordini di grandezza. Cercando quel millimetro tra un modello che non impare ed uno che brilla.
- Il metodo hp.Choice consente di scegliere tra una lista predefinita di valori discreti, utile per decidere tra diversi ottimizzatori o funzioni di attivazione. Scegliamo tra alternative nette, come un tipo di attivazione o un ottimizzatore specifico 
- L'intervallo logaritmico per il learning rate ci permette di campionare valori che variano tra diverse potenze di dieci

Per fare ciò dobbiamo scrivere una funzione di costruzione
La Funzione Model-Building.
Non definiamo più una statua di marmo, ma un modello in argilla che l'oggetto hp può plasmare ad ogni tentativo.
Per usare Keras Tuner dobbiamo incapsulare il nostro modello in una funzione che accetta l'argomento hp. Questo oggetto guida la creazione di ogni variante del modello durante la ricerca.
Possiamo rendere dinamico il numero di layer nascosti usando un ciclo for controllato da un iper-parametro intero, esplorando archiretture più o meno profonde.
E' fondamentale che all'interno della funzione il modello venga compilato con il learning rate suggerito dall'oggetto hp per garantire che ogni prova sia coerente.

Pensate agli iperparametri come alle manopole di una radio: piccoli aggiustamenti possono trasformare il rumore di fondo in un segnale cristallino.
Impostare rante corretti è cruciale. Se l'intervallo è troppo stretto rischiamo di ignorare la soluzione ottima, se è troppo largo la ricerca diventerà inutilmente costosa in termini di tempo CPU.
L'automazione non ci esime dall'essere dei bravi ingenieri, dobbiamo utilizzare l'esperienza per guidare l'algoritmo dove è più probabile si trovi il successo. 

Algoritmi per esplorare lo spazio delle soluzioni.
Una volta definito cosa cercare, dobbiamo decidere come cercare. Non tutte le strategie di esplorazione sono uguali in termini di efficienza e intelligenza.

Abbiamo due modi per la ricerca automatica, differenza tra fortuna ed intelligenza.
- Il RandomSearch seleziona combinazioni casuali di parametri. Sorprendentemente, è spesso più efficace della ricerca a griglia perchè non rimane bloccato su dimensioni meno influenti. Lancia compioni a caso nello spazio, non ha pregiudizi. 
Il RandomSearch è come lanciare freccette bendati: se ne lanci abbastanza prima o poi colpisci il centro. E' perfetto quando abbiamo molta potenza di calcolo parallela.
- L'ottimizzazione Bayesiana è come un esperto cercatore d'oro. Guarda dove ha già scavato ed usa la statistica per prevedere dove sia più probabile trova la prossima petita. Fa molte meno prova 'inutili' per arrivare alla meta.


Ogni ricerca ha un costo.
Max Trials identifica quante combinazioni possiamo permetterci di provare prima di dover dare una risposta.
Execution Per Trial, addestriamo lo stesso modello più volte per assicurarci che un buon risultato non sia solo un colpo di fortuna.
TensorFlow salva tutto in una direttory di progetto, in questo modo abbiamo controllo sul processo di ottimizzazione.


La ricerca è finita, abbiamo i log pieni di dati, è il momento di proclamare il vincitore.
La ricerca è terminata e abbiamo accumulato gigabyte di log. Ora dobbiamo estrarre il 'vincitore' e verificare che le sue promesse siano mantenute anche su dati mai visti.


Identificare il miglior candidato.
- Il metodo 'tuner.results_summary() fornisce una classifica delle migliori combinazioni trovate, mostrando i valori specifici di neuroni e learning rate per ogni prova.
Esso fornisce una classifica delle migliori combinazioni trovate, mostrando i valori specifici di neuroni e learning rate per ogni prova.
- Il metodo 'tuner.get_best_hyperparameters possiamo riprendere una rete già addestrate


Ma la vittoria non è l'ultimo passo del processo
Finalizzazone del modello.
Retraining Finale
Una volta trovato il set di parametri perfetti è buona norma riaddestrare il modello migliore sull'intero dataset (inclusa la validazione), prima del test finale.
Durante il tuning abbiamo tenuto da parte una parte dei dati di tuning per la validazione del tuning.


Evitare l'overfitting dello spazio di ricerca.
Attenzione: se proviamo migliaia di modelli, potremmo trovarne uno che va bene sulla validazione per un puro caso. Questo si chiama 'overfitting' del set di validazione.
Per questo motivo, la valutazione finale (il verdetto finale) deve essere eseguita sempre su un set di test completamente indipendente, non deve mai arrivare dai dati di validazione usati nel tuned, serve un set di test sacro mai visto dalla rete.

In [6]:
import tensorflow as tf
import keras
import keras_tuner as kt #da installare a parte con pip install keras_tuner
import numpy as np

# 1. PREPARAZIONE DEI DATI
# Utilizziamo MNIST, normalizzando i pixel tra 0 e 1 per favorire la convergenza
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0 #per normalizzare
x_test = x_test.astype("float32") / 255.0

# 2. FUNZIONE DI COSTRUZIONE DEL MODELLO (Hypermodel)
# Questa funzione definisce lo "spazio di ricerca": quali parametri testare?
def build_model(hp):
    model = keras.Sequential([
        # Definiamo l'input in modo esplicito (Standard Keras 3)
        keras.Input(shape=(28, 28)), #matrici delle immagini 28x28
        keras.layers.Flatten(),
        
        # TUNING DEI NEURONI: hp.Int crea un range di interi
        # Il Tuner proverà valori come 32, 64, 96... fino a 256  (step di 32)
        keras.layers.Dense(
            units=hp.Int('units', min_value=32, max_value=256, step=32),
            activation='relu'
        ),
        
        keras.layers.Dense(10, activation='softmax')
    ])
    
    # TUNING DEL LEARNING RATE: hp.Float con scala logaritmica
    # La scala logaritmica è ideale per il LR perché esplora ordini di grandezza diversi
    learning_rate = hp.Float('lr', min_value=1e-4, max_value=1e-2, sampling='log')
    
    #compile per costruire la rete neurale, con il learning rate che viene scelto dal tuner
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# 3. INIZIALIZZAZIONE DEL TUNER BAYESIANO
# A differenza della ricerca casuale, l'ottimizzazione Bayesiana usa la statistica 
# per prevedere quali combinazioni di parametri funzioneranno meglio basandosi sui risultati passati.
tuner = kt.BayesianOptimization(
    hypermodel=build_model,
    objective='val_accuracy', # Vogliamo massimizzare la precisione sul set di validazione
    max_trials=5,             # Numero massimo di modelli differenti da testare, limitiamo il numero di prove che può fare
    directory='tuning_results',
    project_name='mnist_keras_2025'
)

# 4. ESECUZIONE DELLA RICERCA (Search), ricerca degli iperparametri migliori.
# Funziona esattamente come il .fit() di Keras. Il Tuner gestisce i cicli di addestramento.
print("--- Inizio ricerca iperparametri ---")
tuner.search(
    x_train, y_train, 
    epochs=3, #ogni modello addestrato su 3 epoche
    validation_split=0.2,
    verbose=1
)

# 5. ESTRAZIONE DEI RISULTATI MIGLIORI
# Recuperiamo i parametri che hanno ottenuto la 'val_accuracy' più alta
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]  #migliori iperparametri trovati

print("\n--- RISULTATI DELLA RICERCA ---")
print(f"Numero ottimale di neuroni: {best_hps.get('units')}")
print(f"Learning Rate ottimale: {best_hps.get('lr'):.5f}")

# Recuperiamo il modello "vincitore" già pronto per l'uso
best_model = tuner.get_best_models(num_models=1)[0]   #modello con i migliori iperparametri

# Valutazione finale sul test set
loss, accuracy = best_model.evaluate(x_test, y_test, verbose=0)  #valutazione del modello sul test set
print(f"Accuratezza finale sul test set: {accuracy:.4f}")

Reloading Tuner from tuning_results\mnist_keras_2025\tuner0.json
--- Inizio ricerca iperparametri ---

--- RISULTATI DELLA RICERCA ---
Numero ottimale di neuroni: 160
Learning Rate ottimale: 0.00067


c:\Users\uberti\.conda\envs\ai_epicode\Lib\site-packages\keras\src\saving\saving_lib.py:801: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Accuratezza finale sul test set: 0.9693
